In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

DATA_PATH = "/content/drive/MyDrive/AKI Project"
# DATA_PATH = "/Users/shruti/Desktop/AMS 585/trinetx_eGFR_gt20_95KPatients"

os.listdir(DATA_PATH)

['patient.csv',
 'procedure.csv',
 'diagnosis.csv',
 'encounter.csv',
 'lab_result.csv',
 'patient_cohort.csv',
 'tumor_properties.csv',
 'medication_drug.csv',
 'oncology_treatment.csv',
 'genomic.csv',
 'chemo_lines.csv',
 'dataset_details.csv',
 'medication_ingredient.csv',
 'tumor.csv',
 'vitals_signs.csv',
 'cohort_details.csv',
 'manifest.csv',
 'standardized_terminology.csv',
 "Kalyani's work.gdoc",
 'AKI_Labs_Baselines.ipynb',
 'Progress report.gdoc',
 'AKI_Project_afterLabs_new.ipynb',
 'AKI_Project_afterLabs.ipynb']

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [4]:
diag_path = os.path.join(DATA_PATH, "diagnosis.csv")

diagnosis = pd.read_csv(diag_path)

diagnosis.head()

,patient_id,encounter_id,code_system,code,principal_diagnosis_indicator,admitting_diagnosis,reason_for_visit,date,derived_by_TriNetX,source_id
0,fQ,iQB,ICD-10-CM,B07.8,P,U,U,20190701,F,EHR
1,fQ,iQN,ICD-10-CM,B07.8,P,U,U,20190708,F,EHR
2,fQ,iQB,ICD-10-CM,B36.0,S,U,U,20190701,F,EHR
3,fQ,iga,ICD-10-CM,B44.9,S,U,U,20231026,F,EHR
4,fQ,iwY,ICD-10-CM,B44.9,P,U,U,20231102,F,EHR


In [5]:
diagnosis.shape
diagnosis['code_system'].value_counts()

,count
code_system,
ICD-10-CM,47978634
ICD-9-CM,16448262


In [6]:
# AKI ICD-10 (N17*)
aki_icd10 = diagnosis[
    (diagnosis['code_system'] == 'ICD-10-CM') &
    (diagnosis['code'].str.startswith('N17', na=False))
]

# AKI ICD-9 (584*)
aki_icd9 = diagnosis[
    (diagnosis['code_system'] == 'ICD-9-CM') &
    (diagnosis['code'].str.startswith('584', na=False))
]

print("AKI ICD-10 count:", aki_icd10.shape[0])
print("AKI ICD-9 count:", aki_icd9.shape[0])

AKI ICD-10 count: 248770
AKI ICD-9 count: 57538


In [7]:
# Combine ICD-10 and ICD-9 AKI rows
aki_all = pd.concat([aki_icd10, aki_icd9])

print("Total AKI diagnosis rows:", aki_all.shape[0])

Total AKI diagnosis rows: 306308


In [8]:
aki_all['date'] = pd.to_datetime(aki_all['date'], format='%Y%m%d', errors='coerce')

aki_all.head()

,patient_id,encounter_id,code_system,code,principal_diagnosis_indicator,admitting_diagnosis,reason_for_visit,date,derived_by_TriNetX,source_id
1651,HRC,KhVC,ICD-10-CM,N17.9,Unknown,U,U,2009-02-14,F,EHR
2425,HBD,KBgD,ICD-10-CM,N17.9,Unknown,U,U,2010-03-27,F,EHR
2426,HBD,KhfD,ICD-10-CM,N17.9,Unknown,U,U,2010-03-28,F,EHR
2910,HhD,KxxD,ICD-10-CM,N17.9,Unknown,U,U,2011-04-11,F,EHR
3140,HRE,KRUE,ICD-10-CM,N17.9,Unknown,U,U,2021-01-08,F,EHR


In [9]:
aki_index = (
    aki_all
    .sort_values('date')
    .groupby('patient_id')
    .first()
    .reset_index()[['patient_id', 'date']]
)

aki_index.rename(columns={'date': 'aki_index_date'}, inplace=True)

print("Number of unique AKI patients:", aki_index.shape[0])
aki_index.head()

Number of unique AKI patients: 33323


,patient_id,aki_index_date
0,#A#B,2016-05-23
1,#A#C,2021-10-16
2,#A#D,2022-07-01
3,#A0B,2018-04-16
4,#A0C,2012-10-18


# EXCLUDING ESRD PATIENTS DIAGNOSED BEFORE AKI

Patients diagnosed with ESRD before or on the AKI index date are excluded because:

- ESRD represents the outcome (end-stage kidney failure).
- The study aims to predict **future** progression to ESRD after AKI.
- Including prior ESRD would violate temporal order.
- It would introduce data leakage and artificially inflate model performance.

This ensures a clean and clinically valid AKI progression cohort.

In [10]:
# ESRD ICD-10 (exact match)
esrd_icd10 = diagnosis[
    (diagnosis['code_system'] == 'ICD-10-CM') &
    (diagnosis['code'] == 'N18.6')
]

# ESRD ICD-9 (exact match)
esrd_icd9 = diagnosis[
    (diagnosis['code_system'] == 'ICD-9-CM') &
    (diagnosis['code'] == '585.6')
]

esrd_all = pd.concat([esrd_icd10, esrd_icd9])

print("Total ESRD diagnosis rows:", esrd_all.shape[0])

Total ESRD diagnosis rows: 308031


In [11]:
# Ensure AKI index date is datetime
aki_index['aki_index_date'] = pd.to_datetime(aki_index['aki_index_date'], errors='coerce')

# Ensure ESRD date is datetime
esrd_all['date'] = pd.to_datetime(esrd_all['date'], format='%Y%m%d', errors='coerce')

In [12]:
esrd_in_aki = esrd_all.merge(aki_index, on='patient_id', how='inner')
print("ESRD rows among AKI patients:", esrd_in_aki.shape[0])
print("AKI patients with ANY ESRD at any time:", esrd_in_aki['patient_id'].nunique())

ESRD rows among AKI patients: 197254
AKI patients with ANY ESRD at any time: 4725


In [13]:
pre_existing_esrd = esrd_in_aki[esrd_in_aki['date'] <= esrd_in_aki['aki_index_date']]

print("Pre-existing ESRD rows (<= index):", pre_existing_esrd.shape[0])
print("Patients with pre-existing ESRD:", pre_existing_esrd['patient_id'].nunique())

Pre-existing ESRD rows (<= index): 34445
Patients with pre-existing ESRD: 1924


In [14]:
exclude_patients = set(pre_existing_esrd['patient_id'].unique())

aki_clean = aki_index[~aki_index['patient_id'].isin(exclude_patients)].copy()

print("AKI patients before exclusion:", aki_index.shape[0])
print("AKI patients after ESRD exclusion:", aki_clean.shape[0])

AKI patients before exclusion: 33323
AKI patients after ESRD exclusion: 31399


In [15]:
print("Any excluded patients still in cohort?",
      aki_clean['patient_id'].isin(exclude_patients).any())

Any excluded patients still in cohort? False


# Defining 2 year prediction window

### index date and end date for each patient

In [16]:
aki_clean['prediction_end_date'] = aki_clean['aki_index_date'] + pd.DateOffset(years=2)

aki_clean.head()

,patient_id,aki_index_date,prediction_end_date
0,#A#B,2016-05-23,2018-05-23
1,#A#C,2021-10-16,2023-10-16
2,#A#D,2022-07-01,2024-07-01
3,#A0B,2018-04-16,2020-04-16
4,#A0C,2012-10-18,2014-10-18


In [17]:
aki_clean.shape[0]

31399

In [18]:
# Merge ESRD with clean AKI cohort
esrd_future = esrd_all.merge(aki_clean, on='patient_id', how='inner')

# Keep ESRD that happens AFTER index and WITHIN 2 years
esrd_future = esrd_future[
    (esrd_future['date'] > esrd_future['aki_index_date']) &
    (esrd_future['date'] <= esrd_future['prediction_end_date'])
]

print("Patients who progressed to ESRD within 2 years:",
      esrd_future['patient_id'].nunique())

Patients who progressed to ESRD within 2 years: 1721


In [19]:
# creating a supervised learning target
# creating outcome label: 1 if progressed to ESRD within 2 years, else 0
esrd_patients_2yr = set(esrd_future['patient_id'].unique())

aki_clean['Y_esrd_2yr'] = aki_clean['patient_id'].isin(esrd_patients_2yr).astype(int)

print(aki_clean['Y_esrd_2yr'].value_counts())
print("Positive rate:", aki_clean['Y_esrd_2yr'].mean())

Y_esrd_2yr
0    29678
1     1721
Name: count, dtype: int64
Positive rate: 0.05481066275996051


Loading patient.csv to inspect code systems

In [20]:
proc_path = os.path.join(DATA_PATH, "procedure.csv")

# Use dtype=str so codes like '585.6' or codes with leading zeros don't get mangled
procedure = pd.read_csv(proc_path, dtype=str)

print(procedure.shape)
print(procedure['code_system'].value_counts())

procedure.head()

(44334752, 8)
code_system
CPT           35371873
HCPCS          6120985
SNOMED         1776498
ICD-10-PCS      570581
ICD-9-CM        494815
Name: count, dtype: int64


,patient_id,encounter_id,code_system,code,principal_procedure_indicator,date,derived_by_TriNetX,source_id
0,fQ,iQ,CPT,11102,Unknown,20220831,F,EHR
1,fQ,ig,CPT,11102,Unknown,20230919,F,EHR
2,fQ,iQ,CPT,11103,Unknown,20220831,F,EHR
3,fQ,iw,CPT,11200,Unknown,20151007,F,EHR
4,fQ,iAB,CPT,15852,Unknown,20221130,F,EHR


In [21]:
# =========================
# DIALYSIS outcome within 2 years
# =========================

# 0) Ensure datetime columns exist
aki_clean = aki_clean.copy()
aki_clean["aki_index_date"] = pd.to_datetime(aki_clean["aki_index_date"], errors="coerce")
if "prediction_end_date" not in aki_clean.columns:
    aki_clean["prediction_end_date"] = aki_clean["aki_index_date"] + pd.DateOffset(years=2)

procedure = procedure.copy()
procedure["date"] = pd.to_datetime(procedure["date"], format="%Y%m%d", errors="coerce")

# 1) Dialysis code definitions (common starters; you can expand later)
CPT_DIALYSIS = {"90935", "90937", "90945", "90947"}      # dialysis procedures
HCPCS_DIALYSIS = {"G0257"}                               # unscheduled dialysis (common)
ICD10PCS_PREFIXES = ("5A1D",)                            # hemodialysis (ICD-10-PCS)
ICD9PROC_DIALYSIS = {"39.95"}                            # hemodialysis (ICD-9 procedure)

# normalize code_system to avoid case issues
procedure["code_system_norm"] = procedure["code_system"].astype(str).str.upper().str.strip()
procedure["code_norm"] = procedure["code"].astype(str).str.strip()

In [22]:
# 2) Pull dialysis rows from procedure table
dialysis_cpt = procedure[
    (procedure["code_system_norm"].str.contains("CPT", na=False)) &
    (procedure["code_norm"].isin(CPT_DIALYSIS))
]

dialysis_hcpcs = procedure[
    (procedure["code_system_norm"].str.contains("HCPCS", na=False)) &
    (procedure["code_norm"].isin(HCPCS_DIALYSIS))
]

dialysis_icd10pcs = procedure[
    (procedure["code_system_norm"].str.contains("ICD-10", na=False)) &
    (procedure["code_system_norm"].str.contains("PCS", na=False)) &
    (procedure["code_norm"].str.startswith(ICD10PCS_PREFIXES, na=False))
]

dialysis_icd9proc = procedure[
    (procedure["code_system_norm"].str.contains("ICD-9", na=False)) &
    (procedure["code_norm"].isin(ICD9PROC_DIALYSIS))
]

dialysis_all = pd.concat(
    [dialysis_cpt, dialysis_hcpcs, dialysis_icd10pcs, dialysis_icd9proc],
    ignore_index=True
)

print("Total dialysis procedure rows (all patients):", dialysis_all.shape[0])

Total dialysis procedure rows (all patients): 100002


In [23]:
# 3) Restrict to AKI cohort and 2-year window
dialysis_in_aki = dialysis_all.merge(
    aki_clean[["patient_id", "aki_index_date", "prediction_end_date"]],
    on="patient_id",
    how="inner"
)

dialysis_future = dialysis_in_aki[
    (dialysis_in_aki["date"] >= dialysis_in_aki["aki_index_date"]) &
    (dialysis_in_aki["date"] <= dialysis_in_aki["prediction_end_date"])
].copy()

dialysis_patients_2yr = set(dialysis_future["patient_id"].unique())
print("AKI patients with ANY dialysis within 2 years (incl acute):", len(dialysis_patients_2yr))

AKI patients with ANY dialysis within 2 years (incl acute): 2448


In [24]:
# 4) Create dialysis label
aki_clean["Y_dialysis_2yr"] = aki_clean["patient_id"].isin(dialysis_patients_2yr).astype(int)
print(aki_clean["Y_dialysis_2yr"].value_counts())
print("Dialysis positive rate:", aki_clean["Y_dialysis_2yr"].mean())

# 5) Combine ESRD OR dialysis label (what Kalyani did)
# (Assumes you already created Y_esrd_2yr; recreate esrd_patients_2yr if you need)
esrd_patients_2yr = set(aki_clean.loc[aki_clean["Y_esrd_2yr"] == 1, "patient_id"])
combined_patients_2yr = esrd_patients_2yr | dialysis_patients_2yr

aki_clean["Y_esrd_or_dialysis_2yr"] = aki_clean["patient_id"].isin(combined_patients_2yr).astype(int)
print(aki_clean["Y_esrd_or_dialysis_2yr"].value_counts())
print("Combined positive rate:", aki_clean["Y_esrd_or_dialysis_2yr"].mean())

Y_dialysis_2yr
0    28951
1     2448
Name: count, dtype: int64
Dialysis positive rate: 0.07796426637791012
Y_esrd_or_dialysis_2yr
0    28208
1     3191
Name: count, dtype: int64
Combined positive rate: 0.10162744036434282


# Building the BLANKING & OBSERVATION windows

In [25]:
# Define a 90-day blanking window before AKI; use all earlier history as input
aki_clean["blank_start_date"] = aki_clean["aki_index_date"] - pd.Timedelta(days=90)

In [26]:
# check difference between index and obs_start
diff_days = (aki_clean["aki_index_date"] - aki_clean["blank_start_date"]).dt.days

print(diff_days.describe())

count    31399.0
mean        90.0
std          0.0
min         90.0
25%         90.0
50%         90.0
75%         90.0
max         90.0
dtype: float64


In [27]:
aki_clean.head()

,patient_id,aki_index_date,prediction_end_date,Y_esrd_2yr,Y_dialysis_2yr,Y_esrd_or_dialysis_2yr,blank_start_date
0,#A#B,2016-05-23,2018-05-23,0,0,0,2016-02-23
1,#A#C,2021-10-16,2023-10-16,1,1,1,2021-07-18
2,#A#D,2022-07-01,2024-07-01,0,0,0,2022-04-02
3,#A0B,2018-04-16,2020-04-16,0,0,0,2018-01-16
4,#A0C,2012-10-18,2014-10-18,0,0,0,2012-07-20


In [28]:
aki_clean.shape

(31399, 7)

In [29]:
# Filter diagnosis events to full history before the 90-day blanking window
diagnosis["date"] = pd.to_datetime(diagnosis["date"], format="%Y%m%d", errors="coerce")

diag_obs = diagnosis.merge(
    aki_clean[["patient_id", "blank_start_date"]],
    on="patient_id",
    how="inner"
)

diag_obs = diag_obs[
    diag_obs["date"] < diag_obs["blank_start_date"]
]

In [30]:
diag_obs.head(20)

,patient_id,encounter_id,code_system,code,principal_diagnosis_indicator,admitting_diagnosis,reason_for_visit,date,derived_by_TriNetX,source_id,blank_start_date
263,HhD,KhME,ICD-10-CM,I37.1,Unknown,U,U,2010-05-10,F,EHR,2011-01-11
268,HhD,KhME,ICD-10-CM,K73.2,Unknown,U,U,2010-05-10,F,EHR,2011-01-11
272,HhD,KhME,ICD-10-CM,Q24.4,Unknown,U,U,2010-05-10,F,EHR,2011-01-11
273,HhD,KRNE,ICD-10-CM,Q24.4,Unknown,U,U,2011-01-10,F,EHR,2011-01-11
310,HhD,KhME,ICD-10-CM,N19,Unknown,U,U,2010-05-10,F,EHR,2011-01-11
321,HhD,KhME,ICD-10-CM,I34.0,Unknown,U,U,2010-05-10,F,EHR,2011-01-11
350,HhD,KhME,ICD-10-CM,I50.9,Unknown,U,U,2010-05-10,F,EHR,2011-01-11
454,HRE,KBVE,ICD-10-CM,J18.9,Unknown,U,U,2013-02-19,F,EHR,2020-10-10
455,HRE,KBVE,ICD-10-CM,R05.9,Unknown,U,U,2013-02-19,F,EHR,2020-10-10
456,HxE,KRfE,ICD-10-CM,A04.72,Unknown,U,U,2022-03-29,F,EHR,2022-05-19


In [31]:
diag_obs["code"].nunique()

29491

# Tabular feature engineering (simpler baseline approach)
- To make sure the cohort is correct
- to check if signals/indicators exist
- gives a benchmark model

Later I will progress to sequences like Kalyani did so I could compare the improvement over this simple baseline

In [32]:
# finding the 300 top most frequent diagnoses codes and arranging them in a table with the codes as columns (300 columns)

top_codes = (
    diag_obs["code"]
    .value_counts()
    .head(300)
    .index
)

In [33]:
diag_obs_top = diag_obs[diag_obs["code"].isin(top_codes)]

In [34]:
diag_features = (
    diag_obs_top
    .groupby(["patient_id", "code"])
    .size()
    .unstack(fill_value=0)
)

In [35]:
diag_features.head()

code,042,110.1,162.9,174.9,185,203.00,211.3,244.9,250.00,250.02,250.40,250.50,250.60,268.9,272.0,272.2,272.4,274.9,276.8,278.00,278.01,280.9,285.9,300.00,305.1,311,327.23,338.29,357.2,366.16,401.1,401.9,414.00,414.01,416.8,424.1,425.4,427.31,427.89,428.0,434.91,443.9,465.9,477.9,486,493.90,496,518.89,530.81,562.10,564.00,571.5,585.3,585.9,592.0,593.9,599.0,600.00,714.0,715.16,715.90,715.96,719.41,719.45,719.46,721.3,722.52,723.1,724.02,724.2,724.4,724.5,729.1,729.5,733.00,733.90,780.2,780.4,780.57,780.79,782.3,784.0,786.05,786.09,786.2,786.50,786.59,787.91,789.00,790.29,790.6,794.31,B18.2,B20,B35.1,C18.9,C20,C22.0,C34.90,C50.9,C56.9,C61,C67.9,C79.51,C90.00,C92.00,D46.9,D50.9,D64.9,D69.6,E03.9,E11.22,E11.40,E11.42,E11.621,E11.65,E11.69,E11.8,E11.9,E53.8,E55.9,E66.01,E66.9,E78.00,E78.2,E78.5,E83.42,E87.1,E87.6,F10.10,F17.200,F17.210,F31.9,F32.89,F32.9,F32.A,F33.1,F41.1,F41.9,G35,G40.909,G47.00,G47.30,G47.33,G62.9,G89.29,G89.4,H26.9,I10,I11.0,I12.9,I25.10,I25.2,I25.5,I26.99,I27.20,I34.0,I35.0,I42.8,I42.9,I48.0,I48.2,I48.20,I48.91,I48.92,I50.22,I50.30,I50.32,I50.9,I51.7,I63.9,I73.9,I87.2,J06.9,J18.9,J30.9,J44.1,J44.89,J44.9,J45.909,J90,K21.9,K44.9,K57.30,K59.00,K74.60,K76.0,M06.9,M10.9,M17.0,M17.11,M17.12,M19.90,M25.511,M25.512,M25.551,M25.552,M25.561,M25.562,M25.569,M47.816,M48.061,M51.36,M54.16,M54.2,M54.5,M54.50,M54.9,M79.609,M79.89,M81.0,M85.80,N18.3,N18.30,N18.4,N18.9,N20.0,N28.9,N39.0,N40.0,N40.1,R00.0,R00.1,R00.2,R05,R05.9,R06.00,R06.02,R06.09,R07.89,R07.9,R09.02,R09.89,R10.13,R10.9,R11.0,R11.2,R13.10,R19.7,R21,R30.0,R31.9,R33.9,R35.0,R42,R50.9,R51,R52,R53.1,R53.81,R53.83,R55,R56.9,R60.0,R60.9,R63.4,R73.03,R73.9,R79.89,R80.9,R91.1,R91.8,R94.31,T45.1X5A,V04.81,V45.81,V45.89,V58.11,V58.61,V58.69,V58.83,V70.0,V76.12,W19.XXXA,Z00.00,Z01.818,Z09,Z12.11,Z12.31,Z20.822,Z23,Z51.11,Z51.81,Z72.0,Z79.01,Z79.4,Z79.82,Z79.84,Z79.899,Z86.718,Z86.73,Z87.891,Z90.49,Z95.0,Z95.1,Z95.2,Z95.5,Z95.810,Z96.1,Z98.890
patient_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
#A#B,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,3,0,0,0,2,2,0,2,0,0,0,2,1,0,0,0,1,4,1,0,2,0,1,0,0,0,0,13,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,1,0,1,1,0,5,0,1,0,0,0,0,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
#A#C,0,0,0,0,0,0,0,0,5,0,0,0,0,0,9,0,0,0,0,1,0,0,0,2,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,10,0,0,1,14,0,7,0,0,0,21,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,2,0,0,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,2,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,2,0,0,0,0,1,5,0,0,1,1,0,0,0,2,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,6,0,0,11,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
#A#D,0,0,0,0,0,0,0,0,35,18,0,0,0,0,0,15,15,0,1,0,0,0,0,0,0,0,0,0,0,0,38,1,13,1,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,7,0,0,1,0,0,0,0,0,14,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,20,0,0,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,1,0,0,0,0,0,0,0,0,0

In [36]:
# merging with the AKI cohort
X = aki_clean[["patient_id"]].merge(
    diag_features,
    on="patient_id",
    how="left"
)

X = X.fillna(0)

In [37]:
X.head()

,patient_id,042,110.1,162.9,174.9,185,203.00,211.3,244.9,250.00,250.02,250.40,250.50,250.60,268.9,272.0,272.2,272.4,274.9,276.8,278.00,278.01,280.9,285.9,300.00,305.1,311,327.23,338.29,357.2,366.16,401.1,401.9,414.00,414.01,416.8,424.1,425.4,427.31,427.89,428.0,434.91,443.9,465.9,477.9,486,493.90,496,518.89,530.81,562.10,564.00,571.5,585.3,585.9,592.0,593.9,599.0,600.00,714.0,715.16,715.90,715.96,719.41,719.45,719.46,721.3,722.52,723.1,724.02,724.2,724.4,724.5,729.1,729.5,733.00,733.90,780.2,780.4,780.57,780.79,782.3,784.0,786.05,786.09,786.2,786.50,786.59,787.91,789.00,790.29,790.6,794.31,B18.2,B20,B35.1,C18.9,C20,C22.0,C34.90,C50.9,C56.9,C61,C67.9,C79.51,C90.00,C92.00,D46.9,D50.9,D64.9,D69.6,E03.9,E11.22,E11.40,E11.42,E11.621,E11.65,E11.69,E11.8,E11.9,E53.8,E55.9,E66.01,E66.9,E78.00,E78.2,E78.5,E83.42,E87.1,E87.6,F10.10,F17.200,F17.210,F31.9,F32.89,F32.9,F32.A,F33.1,F41.1,F41.9,G35,G40.909,G47.00,G47.30,G47.33,G62.9,G89.29,G89.4,H26.9,I10,I11.0,I12.9,I25.10,I25.2,I25.5,I26.99,I27.20,I34.0,I35.0,I42.8,I42.9,I48.0,I48.2,I48.20,I48.91,I48.92,I50.22,I50.30,I50.32,I50.9,I51.7,I63.9,I73.9,I87.2,J06.9,J18.9,J30.9,J44.1,J44.89,J44.9,J45.909,J90,K21.9,K44.9,K57.30,K59.00,K74.60,K76.0,M06.9,M10.9,M17.0,M17.11,M17.12,M19.90,M25.511,M25.512,M25.551,M25.552,M25.561,M25.562,M25.569,M47.816,M48.061,M51.36,M54.16,M54.2,M54.5,M54.50,M54.9,M79.609,M79.89,M81.0,M85.80,N18.3,N18.30,N18.4,N18.9,N20.0,N28.9,N39.0,N40.0,N40.1,R00.0,R00.1,R00.2,R05,R05.9,R06.00,R06.02,R06.09,R07.89,R07.9,R09.02,R09.89,R10.13,R10.9,R11.0,R11.2,R13.10,R19.7,R21,R30.0,R31.9,R33.9,R35.0,R42,R50.9,R51,R52,R53.1,R53.81,R53.83,R55,R56.9,R60.0,R60.9,R63.4,R73.03,R73.9,R79.89,R80.9,R91.1,R91.8,R94.31,T45.1X5A,V04.81,V45.81,V45.89,V58.11,V58.61,V58.69,V58.83,V70.0,V76.12,W19.XXXA,Z00.00,Z01.818,Z09,Z12.11,Z12.31,Z20.822,Z23,Z51.11,Z51.81,Z72.0,Z79.01,Z79.4,Z79.82,Z79.84,Z79.899,Z86.718,Z86.73,Z87.891,Z90.49,Z95.0,Z95.1,Z95.2,Z95.5,Z95.810,Z96.1,Z98.890
0,#A#B,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,0.0,1.0,4.0,1.0,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,0.0,1.0,1.0,0.0,5.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,#A#C,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,1.0,14.0,0.0,7.0,0.0,0.0,0.0,21.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0

In [38]:
# Remove old target if it exists
if "target" in X.columns:
    X = X.drop(columns=["target"])

In [39]:
# Add target safely by merging on patient_id
X = X.merge(
    aki_clean[["patient_id", "Y_esrd_or_dialysis_2yr"]],
    on="patient_id",
    how="left"
)

X = X.rename(columns={"Y_esrd_or_dialysis_2yr": "target"})

In [40]:
X.head()

,patient_id,042,110.1,162.9,174.9,185,203.00,211.3,244.9,250.00,250.02,250.40,250.50,250.60,268.9,272.0,272.2,272.4,274.9,276.8,278.00,278.01,280.9,285.9,300.00,305.1,311,327.23,338.29,357.2,366.16,401.1,401.9,414.00,414.01,416.8,424.1,425.4,427.31,427.89,428.0,434.91,443.9,465.9,477.9,486,493.90,496,518.89,530.81,562.10,564.00,571.5,585.3,585.9,592.0,593.9,599.0,600.00,714.0,715.16,715.90,715.96,719.41,719.45,719.46,721.3,722.52,723.1,724.02,724.2,724.4,724.5,729.1,729.5,733.00,733.90,780.2,780.4,780.57,780.79,782.3,784.0,786.05,786.09,786.2,786.50,786.59,787.91,789.00,790.29,790.6,794.31,B18.2,B20,B35.1,C18.9,C20,C22.0,C34.90,C50.9,C56.9,C61,C67.9,C79.51,C90.00,C92.00,D46.9,D50.9,D64.9,D69.6,E03.9,E11.22,E11.40,E11.42,E11.621,E11.65,E11.69,E11.8,E11.9,E53.8,E55.9,E66.01,E66.9,E78.00,E78.2,E78.5,E83.42,E87.1,E87.6,F10.10,F17.200,F17.210,F31.9,F32.89,F32.9,F32.A,F33.1,F41.1,F41.9,G35,G40.909,G47.00,G47.30,G47.33,G62.9,G89.29,G89.4,H26.9,I10,I11.0,I12.9,I25.10,I25.2,I25.5,I26.99,I27.20,I34.0,I35.0,I42.8,I42.9,I48.0,I48.2,I48.20,I48.91,I48.92,I50.22,I50.30,I50.32,I50.9,I51.7,I63.9,I73.9,I87.2,J06.9,J18.9,J30.9,J44.1,J44.89,J44.9,J45.909,J90,K21.9,K44.9,K57.30,K59.00,K74.60,K76.0,M06.9,M10.9,M17.0,M17.11,M17.12,M19.90,M25.511,M25.512,M25.551,M25.552,M25.561,M25.562,M25.569,M47.816,M48.061,M51.36,M54.16,M54.2,M54.5,M54.50,M54.9,M79.609,M79.89,M81.0,M85.80,N18.3,N18.30,N18.4,N18.9,N20.0,N28.9,N39.0,N40.0,N40.1,R00.0,R00.1,R00.2,R05,R05.9,R06.00,R06.02,R06.09,R07.89,R07.9,R09.02,R09.89,R10.13,R10.9,R11.0,R11.2,R13.10,R19.7,R21,R30.0,R31.9,R33.9,R35.0,R42,R50.9,R51,R52,R53.1,R53.81,R53.83,R55,R56.9,R60.0,R60.9,R63.4,R73.03,R73.9,R79.89,R80.9,R91.1,R91.8,R94.31,T45.1X5A,V04.81,V45.81,V45.89,V58.11,V58.61,V58.69,V58.83,V70.0,V76.12,W19.XXXA,Z00.00,Z01.818,Z09,Z12.11,Z12.31,Z20.822,Z23,Z51.11,Z51.81,Z72.0,Z79.01,Z79.4,Z79.82,Z79.84,Z79.899,Z86.718,Z86.73,Z87.891,Z90.49,Z95.0,Z95.1,Z95.2,Z95.5,Z95.810,Z96.1,Z98.890,target
0,#A#B,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,0.0,1.0,4.0,1.0,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,0.0,1.0,1.0,0.0,5.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,#A#C,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,1.0,14.0,0.0,7.0,0.0,0.0,0.0,21.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.

In [41]:
print(X.shape)                  # should be (31399, 300+2)
print(X["target"].isna().sum()) # should be 0
print(X["target"].value_counts())

(31399, 302)
0
target
0    28208
1     3191
Name: count, dtype: int64


# ML Modeling

In [42]:
# separating features and target
# features
X_model = X.drop(columns=["patient_id", "target"])

# target
y_model = X["target"]

print(X_model.shape)
print(y_model.shape)

(31399, 300)
(31399,)


In [43]:
y_model.isna().sum()

np.int64(0)

In [44]:
# removing nan from target variable
mask = y_model.notna()

X_model = X_model[mask]
y_model = y_model[mask]

print("New shape:", X_model.shape)

New shape: (31399, 300)


In [45]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42,
    stratify=y_model
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (25119, 300)
Test shape: (6280, 300)


In [46]:
y_model.value_counts()

,count
target,
0,28208
1,3191


In [47]:
# starting with LR as it is the standard baseline for classification
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [48]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [49]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.8964968152866242
ROC AUC: 0.6402897714076802
F1 Score: 0.0
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      5642
           1       0.00      0.00      0.00       638

    accuracy                           0.90      6280
   macro avg       0.45      0.50      0.47      6280
weighted avg       0.81      0.90      0.85      6280



### above results show the dataset is heavily imbalanced and the accuracy is misleading.
- ROC AUC being 0.5 means the model is randomly guessing.
- F1 Score: 0.003 (extremely low) because recall=0.0 which means the model almost never predicts dialysis patients

# CLASS IMBALANCE FIX
- applying class weights to fix the class imbalance issue

In [50]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [51]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [52]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.44920382165605094
ROC AUC: 0.6381748118399955
F1 Score: 0.22496078870714767
              precision    recall  f1-score   support

           0       0.94      0.41      0.57      5642
           1       0.13      0.79      0.22       638

    accuracy                           0.45      6280
   macro avg       0.54      0.60      0.40      6280
weighted avg       0.86      0.45      0.54      6280



- this is a big improvement as recall 0.6 which means the model is finally detecting dialysis patients (model finds 60% of real dialysis patients)
- precision = 0.1 (only 10% of predicted dialysis are correct)
- F1 = 0.17 overall balance
- ROC AUC is still bad as the model cannot separate low-risk vs high-risk patients

# NEXT improvements
- adding procedure features
- adding medication features
- adding CKD stage features
- using better models

# REPLICATING KALYANIS PIPELINE

In [53]:
# Filter procedure events to full history before the 90-day blanking window
import pandas as pd
import os

proc_path = os.path.join(DATA_PATH, "procedure.csv")
procedure = pd.read_csv(proc_path, dtype=str)

procedure["date"] = pd.to_datetime(procedure["date"], format="%Y%m%d", errors="coerce")
aki_clean["blank_start_date"] = pd.to_datetime(aki_clean["blank_start_date"], errors="coerce")

proc_obs = procedure.merge(
    aki_clean[["patient_id", "blank_start_date"]],
    on="patient_id",
    how="inner"
)

proc_obs = proc_obs[
    proc_obs["date"] < proc_obs["blank_start_date"]
]

print(proc_obs.shape)
proc_obs.head()

(5567753, 9)


,patient_id,encounter_id,code_system,code,principal_procedure_indicator,date,derived_by_TriNetX,source_id,blank_start_date
605,HxE,Kx6E,CPT,00532,Unknown,2022-02-10,F,EHR,2022-05-19
606,HxE,KB7E,CPT,00635,Unknown,2022-02-20,F,EHR,2022-05-19
607,HxE,KR7E,CPT,00635,Unknown,2022-03-20,F,EHR,2022-05-19
618,HxE,Kx9E,CPT,36415,Unknown,2022-03-17,F,EHR,2022-05-19
619,HxE,KR7E,CPT,36415,Unknown,2022-03-20,F,EHR,2022-05-19


In [54]:
# Filter medication events to full history before the 90-day blanking window
med_path = os.path.join(DATA_PATH, "medication_drug.csv")
# med = pd.read_csv(med_path, dtype=str, on_bad_lines='skip')
med = pd.read_csv(med_path, dtype=str, on_bad_lines='skip', engine='python')

med["start_date"] = pd.to_datetime(med["start_date"], format="%Y%m%d", errors="coerce")
aki_clean["blank_start_date"] = pd.to_datetime(aki_clean["blank_start_date"], errors="coerce")

med_obs = med.merge(
    aki_clean[["patient_id", "blank_start_date"]],
    on="patient_id",
    how="inner"
)

med_obs = med_obs[
    med_obs["start_date"] < med_obs["blank_start_date"]
]

print(med_obs.shape)
med_obs.head()

(5961932, 14)


,patient_id,encounter_id,unique_id,code_system,code,start_date,route,brand,strength,quantity_dispensed,days_supply,derived_by_TriNetX,source_id,blank_start_date
38,HhD,KxfN,Hx7h,RxNorm,310798,2010-03-05,Oral Product,Generic,Unknown,NaN,NaN,F,EHR,2011-01-11
40,HhD,KxsN,HxDi,RxNorm,1654008,2006-10-11,Injectable Product,Generic,Unknown,NaN,NaN,F,EHR,2011-01-11
41,HhD,KxfN,HBEi,RxNorm,1654008,2010-03-08,Injectable Product,Generic,Unknown,NaN,NaN,F,EHR,2011-01-11
42,HhD,KhME,HREi,RxNorm,1654008,2010-05-14,Injectable Product,Generic,Unknown,NaN,NaN,F,EHR,2011-01-11
44,HhD,KxfN,HRFi,RxNorm,1808224,2010-03-08,Injectable Product,Generic,Unknown,NaN,NaN,F,EHR,2011-01-11


In [55]:
# Filter diagnosis events to full history before the 90-day blanking window
diag_obs = diagnosis.merge(
    aki_clean[["patient_id", "blank_start_date"]],
    on="patient_id",
    how="inner"
)

diag_obs = diag_obs[
    diag_obs["date"] < diag_obs["blank_start_date"]
]

diag_obs.head()

,patient_id,encounter_id,code_system,code,principal_diagnosis_indicator,admitting_diagnosis,reason_for_visit,date,derived_by_TriNetX,source_id,blank_start_date
263,HhD,KhME,ICD-10-CM,I37.1,Unknown,U,U,2010-05-10,F,EHR,2011-01-11
268,HhD,KhME,ICD-10-CM,K73.2,Unknown,U,U,2010-05-10,F,EHR,2011-01-11
272,HhD,KhME,ICD-10-CM,Q24.4,Unknown,U,U,2010-05-10,F,EHR,2011-01-11
273,HhD,KRNE,ICD-10-CM,Q24.4,Unknown,U,U,2011-01-10,F,EHR,2011-01-11
310,HhD,KhME,ICD-10-CM,N19,Unknown,U,U,2010-05-10,F,EHR,2011-01-11


In [56]:
print(diag_obs.shape)
print(proc_obs.shape)
print(med_obs.shape)

(8717950, 11)
(5567753, 9)
(5961932, 14)


Meaning:

~8.72M diagnosis events

~5.57M procedure events

~5.96M medication events

All from full patient history before the 90-day blanking window.

Total timeline size: ~20.25M events

In [57]:
# converting each table into same event format

# diagnosis
diagnosis_events = diag_obs[["patient_id","date","code"]].copy()
diagnosis_events["event_type"] = "diagnosis"
diagnosis_events = diagnosis_events.dropna(subset=["date"])

# procedure
procedure_events = proc_obs[["patient_id","date","code"]].copy()
procedure_events["event_type"] = "procedure"
procedure_events = procedure_events.dropna(subset=["date"])

# medication
medication_events = med_obs[["patient_id","start_date","code"]].copy()
medication_events = medication_events.rename(columns={"start_date":"date"})
medication_events["event_type"] = "medication"
medication_events = medication_events.dropna(subset=["date"])

# merge all
all_events = pd.concat(
    [diagnosis_events, procedure_events, medication_events],
    ignore_index=True
)

print(all_events.shape)

(20247635, 4)


In [58]:
# merging all events

all_events = pd.concat(
    [diagnosis_events, procedure_events, medication_events],
    ignore_index=True
)

print(all_events.shape)
all_events.head()

(20247635, 4)


,patient_id,date,code,event_type
0,HhD,2010-05-10,I37.1,diagnosis
1,HhD,2010-05-10,K73.2,diagnosis
2,HhD,2010-05-10,Q24.4,diagnosis
3,HhD,2011-01-10,Q24.4,diagnosis
4,HhD,2010-05-10,N19,diagnosis


In [59]:
# sorting events chronologically per patient
all_events = all_events.sort_values(["patient_id", "date"])

all_events.head(20)

,patient_id,date,code,event_type
1909038,#A#B,2013-01-17,466.0,diagnosis
1909080,#A#B,2013-01-17,786.2,diagnosis
1909081,#A#B,2013-01-17,786.9,diagnosis
1909094,#A#B,2013-01-17,959.01,diagnosis
1909095,#A#B,2013-01-17,959.01,diagnosis
1909096,#A#B,2013-01-17,959.9,diagnosis
1909097,#A#B,2013-01-17,E885.9,diagnosis
1909098,#A#B,2013-01-17,E888.9,diagnosis
1909099,#A#B,2013-01-17,E888.9,diagnosis
10502862,#A#B,2013-01-17,70450,procedure


In [60]:
# removing sparse patients (very short sequences)
# computing sequence length
seq_lengths = all_events.groupby("patient_id").size()

seq_lengths.head()

,0
patient_id,
#A#B,459
#A#C,517
#A#D,1331
#A0B,511
#A0C,64


In [61]:
seq_lengths.shape

(27084,)

In [62]:
# keeping patients with more than 3 events
valid_patients = seq_lengths[seq_lengths >= 3].index

In [63]:
valid_patients.shape

(26357,)

In [64]:
# filtering the timeline
all_events_filtered = all_events[
    all_events["patient_id"].isin(valid_patients)
]

print(all_events_filtered.shape)

(20246598, 4)


In [65]:
# Sorting filtered events by patient and time before sequence creation

all_events_filtered = all_events_filtered.sort_values(
    by=["patient_id", "date"]
).reset_index(drop=True)

In [66]:
# converting the codes into sequence tokens for the transformers
# make each event unique by combining type + code
all_events_filtered["event_token"] = (
    all_events_filtered["event_type"].astype(str) + "_" +
    all_events_filtered["code"].astype(str)
)

all_events_filtered.head()

,patient_id,date,code,event_type,event_token
0,#A#B,2013-01-17,466.0,diagnosis,diagnosis_466.0
1,#A#B,2013-01-17,786.2,diagnosis,diagnosis_786.2
2,#A#B,2013-01-17,786.9,diagnosis,diagnosis_786.9
3,#A#B,2013-01-17,959.01,diagnosis,diagnosis_959.01
4,#A#B,2013-01-17,959.01,diagnosis,diagnosis_959.01


In [67]:
# grouping the events per patient (UPDATED for transformer tokens)
patient_sequences = all_events_filtered.groupby("patient_id")["event_token"].apply(list)

print(patient_sequences.shape)
patient_sequences.head()

(26357,)


,event_token
patient_id,
#A#B,"[diagnosis_466.0, diagnosis_786.2, diagnosis_7..."
#A#C,"[diagnosis_V04.81, procedure_90471, procedure_..."
#A#D,"[diagnosis_250.00, diagnosis_401.1, diagnosis_..."
#A0B,"[diagnosis_729.5, diagnosis_E880.1, procedure_..."
#A0C,"[diagnosis_593.2, procedure_76770, diagnosis_4..."


In [68]:
# build vocabulary and numeric sequences
# unique event tokens
unique_tokens = all_events_filtered["event_token"].unique()

# token -> integer mapping
token2id = {token: i + 1 for i, token in enumerate(unique_tokens)}   # start from 1
id2token = {i: token for token, i in token2id.items()}

print("Vocabulary size:", len(token2id))

Vocabulary size: 75603


In [69]:
# converting each patient timeline into integer sequences
patient_sequences = (
    all_events_filtered
    .groupby("patient_id")["event_token"]
    .apply(list)
)

patient_sequences_ids = patient_sequences.apply(
    lambda seq: [token2id[token] for token in seq]
)

print(patient_sequences_ids.shape)
patient_sequences_ids.head()

(26357,)


,event_token
patient_id,
#A#B,"[1, 2, 3, 4, 4, 5, 6, 7, 7, 8, 9, 10, 11, 12, ..."
#A#C,"[174, 175, 176, 174, 174, 177, 174, 174, 175, ..."
#A#D,"[156, 76, 339, 340, 341, 342, 249, 343, 344, 5..."
#A0B,"[500, 501, 502, 503, 156, 90, 504, 184, 249, 2..."
#A0C,"[582, 583, 584, 585, 586, 587, 12, 352, 12, 29..."


In [70]:
# attaching the label Y to each patient sequence
# create patient-level label table
labels = aki_clean[["patient_id", "Y_esrd_or_dialysis_2yr"]].drop_duplicates()

# turn sequence series into dataframe
seq_df = patient_sequences_ids.reset_index()
seq_df.columns = ["patient_id", "sequence"]

# merge sequences with labels
seq_df = seq_df.merge(labels, on="patient_id", how="inner")

print(seq_df.shape)
seq_df.head()

(26357, 3)


,patient_id,sequence,Y_esrd_or_dialysis_2yr
0,#A#B,"[1, 2, 3, 4, 4, 5, 6, 7, 7, 8, 9, 10, 11, 12, ...",0
1,#A#C,"[174, 175, 176, 174, 174, 177, 174, 174, 175, ...",1
2,#A#D,"[156, 76, 339, 340, 341, 342, 249, 343, 344, 5...",0
3,#A0B,"[500, 501, 502, 503, 156, 90, 504, 184, 249, 2...",0
4,#A0C,"[582, 583, 584, 585, 586, 587, 12, 352, 12, 29...",0


In [71]:
# train test split
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    seq_df,
    test_size=0.2,
    random_state=42,
    stratify=seq_df["Y_esrd_or_dialysis_2yr"]
)

print(train_df.shape, test_df.shape)

(21085, 3) (5272, 3)


In [72]:
# finding the sequence lenghts before padding
train_df["seq_len"] = train_df["sequence"].apply(len)
test_df["seq_len"] = test_df["sequence"].apply(len)

print(train_df["seq_len"].describe())
print(test_df["seq_len"].describe())

count     21085.000000
mean        768.692910
std        1967.776196
min           3.000000
25%         110.000000
50%         364.000000
75%         923.000000
max      199578.000000
Name: seq_len, dtype: float64
count     5272.000000
mean       766.067527
std       1311.154266
min          3.000000
25%        106.000000
50%        353.000000
75%        921.250000
max      41552.000000
Name: seq_len, dtype: float64


In [73]:
import numpy as np

MAX_SEQ_LEN = 400

def fast_pad_sequences(sequences, maxlen, padding='post', truncating='post'):
    out = np.zeros((len(sequences), maxlen), dtype=np.int32)
    for i, seq in enumerate(sequences):
        seq = list(seq)
        if len(seq) > maxlen:
            seq = seq[:maxlen] if truncating == 'post' else seq[-maxlen:]
        if len(seq) > 0:
            if padding == 'post':
                out[i, :len(seq)] = seq
            else:
                out[i, -len(seq):] = seq
    return out

X_train = fast_pad_sequences(train_df["sequence"], maxlen=MAX_SEQ_LEN)
X_test  = fast_pad_sequences(test_df["sequence"],  maxlen=MAX_SEQ_LEN)

y_train = train_df["Y_esrd_or_dialysis_2yr"].values
y_test  = test_df["Y_esrd_or_dialysis_2yr"].values

print(X_train.shape)
print(X_test.shape)

(21085, 400)
(5272, 400)


# Building RETAIN.

In [75]:
import tensorflow as tf
from tensorflow.keras import layers, Model

VOCAB_SIZE   = len(token2id) + 1   # +1 for pad token = 0
MAX_SEQ_LEN  = 400
EMBED_DIM    = 128
ALPHA_HIDDEN = 128
BETA_HIDDEN  = 128
DROPOUT      = 0.1

def build_retain_model(maxlen, vocab_size,
                       embed_dim=128, alpha_hidden=128, beta_hidden=128, rate=0.1):
    """
    RETAIN (Choi et al. 2016) — two-level attention over patient event sequences.
      α : scalar attention over events (which events matter)
      β : vector attention over embedding dims (which features within an event matter)
    Reads sequences in REVERSE so the most recent event seeds the GRU hidden state.
    """
    inputs = layers.Input(shape=(maxlen,), dtype="int32", name="event_ids")

    # mask: 1 for real tokens, 0 for padding (wrap tf ops in Lambda for Keras 3)
    mask_exp = layers.Lambda(
        lambda x: tf.expand_dims(tf.cast(tf.not_equal(x, 0), tf.float32), -1),
        name="mask",
    )(inputs)                                                          # (B, T, 1)

    # event embeddings, with padded positions zeroed out
    v = layers.Embedding(vocab_size, embed_dim, name="event_embedding")(inputs)
    v = layers.Multiply(name="zero_pad_embeds")([v, mask_exp])
    v = layers.Dropout(rate)(v)

    # reverse sequence so most recent event comes first into the GRUs
    v_rev = layers.Lambda(lambda t: tf.reverse(t, axis=[1]))(v)

    # ── α : event-level scalar attention ───────────────────────────────
    g = layers.GRU(alpha_hidden, return_sequences=True, name="alpha_gru")(v_rev)
    alpha_logits = layers.Dense(1, name="alpha_logits")(g)             # (B, T, 1)
    # un-reverse so alpha aligns with original event order
    alpha_logits = layers.Lambda(lambda t: tf.reverse(t, axis=[1]))(alpha_logits)
    # mask padded positions BEFORE softmax (large negative => ~0 weight)
    neg_inf_mask = layers.Lambda(
        lambda m: (1.0 - m) * -1e9, name="neg_inf_mask"
    )(mask_exp)
    alpha_logits = layers.Add()([alpha_logits, neg_inf_mask])
    alpha = layers.Softmax(axis=1, name="alpha")(alpha_logits)         # (B, T, 1)

    # ── β : feature-level vector attention ────────────────────────────
    h = layers.GRU(beta_hidden, return_sequences=True, name="beta_gru")(v_rev)
    beta = layers.Dense(embed_dim, activation="tanh", name="beta")(h)
    beta = layers.Lambda(lambda t: tf.reverse(t, axis=[1]))(beta)      # (B, T, E)

    # context vector: c = Σ_t α_t · (β_t ⊙ v_t)
    weighted = layers.Multiply()([beta, v])
    weighted = layers.Multiply()([alpha, weighted])
    weighted = layers.Multiply()([weighted, mask_exp])                  # belt + suspenders
    context  = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1), name="context")(weighted)
    context  = layers.Dropout(rate)(context)

    outputs = layers.Dense(1, activation="sigmoid", name="output")(context)
    return Model(inputs=inputs, outputs=outputs, name="RETAIN")

model = build_retain_model(
    maxlen=MAX_SEQ_LEN,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    alpha_hidden=ALPHA_HIDDEN,
    beta_hidden=BETA_HIDDEN,
    rate=DROPOUT,
)
model.summary()

Model: "RETAIN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ event_ids           │ (None, 400)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ event_embedding     │ (None, 400, 128)  │  9,677,312 │ event_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask (Lambda)       │ (None, 400, 1)    │          0 │ event_ids[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_pad_embeds     │ (None, 400, 128)  │          0 │ event_embedding[… │
│ (Multiply)          │                   │            │ mask[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 400, 128)  │          0 │ zero_pad_embeds[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 400, 128)  │          0 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ alpha_gru (GRU)     │ (None, 400, 128)  │     99,072 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ alpha_logits        │ (None, 400, 1)    │        129 │ alpha_gru[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ beta_gru (GRU)      │ (None, 400, 128)  │     99,072 │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 400, 1)    │          0 │ alpha_logits[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ neg_inf_mask        │ (None, 400, 1)    │          0 │ mask[0][0]        │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ beta (Dense)        │ (None, 400, 128)  │     16,512 │ beta_gru[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 400, 1)    │          0 │ lambda_1[0][0],   │
│                     │                   │            │ neg_inf_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 400, 128)  │          0 │ beta[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ alpha (Softmax)     │ (None, 400, 1)    │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 400, 128)  │          0 │ lambda_2[0][0],   │
│                     │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 400, 128)  │          0 │ alpha[0][0],      │
│ (Multiply)          │                   │            │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_2          │ (None, 400, 128)  │          0 │ multiply_1[0][0], │
│ (Multiply)          │                   │            │ mask[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context (Lambda)    │ (None, 128)       │          0 │ multiply_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ context[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 9,892,226 (37.74 MB)

 Trainable params: 9,892,226 (37.74 MB)

 Non-trainable params: 0 (0.00 B)

In [76]:
import tensorflow as tf

def binary_focal_loss(alpha=0.75, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)

        loss_val = -alpha_t * tf.pow(1.0 - pt, gamma) * tf.math.log(pt)
        return tf.reduce_mean(loss_val)

    return loss

In [77]:
# compiling focal loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=binary_focal_loss(alpha=0.5, gamma=2.0),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

In [78]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    patience=3,
    mode="max",
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 24s 34ms/step - accuracy: 0.9128 - auc: 0.5677 - loss: 0.0432 - val_accuracy: 0.9115 - val_auc: 0.6289 - val_loss: 0.0396
Epoch 2/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - accuracy: 0.9153 - auc: 0.7979 - loss: 0.0338 - val_accuracy: 0.9082 - val_auc: 0.6301 - val_loss: 0.0408
Epoch 3/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - accuracy: 0.9296 - auc: 0.9105 - loss: 0.0251 - val_accuracy: 0.8990 - val_auc: 0.6029 - val_loss: 0.0494
Epoch 4/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - accuracy: 0.9587 - auc: 0.9741 - loss: 0.0146 - val_accuracy: 0.8916 - val_auc: 0.5878 - val_loss: 0.0766
Epoch 5/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - accuracy: 0.9798 - auc: 0.9937 - loss: 0.0074 - val_accuracy: 0.8767 - val_auc: 0.5874 - val_loss: 0.1017


In [79]:
# evaluate test set
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step
Accuracy: 0.9112291350531108
ROC AUC: 0.6484652765635017
F1 Score: 0.029045643153526972
              precision    recall  f1-score   support

           0       0.91      1.00      0.95      4816
           1       0.27      0.02      0.03       456

    accuracy                           0.91      5272
   macro avg       0.59      0.51      0.49      5272
weighted avg       0.86      0.91      0.87      5272



In [80]:
# adding new metrics:

from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("Sensitivity:", sensitivity)
print("Specificity:", specificity)

Sensitivity: 0.015350877192982455
Specificity: 0.9960548172757475


In [81]:
# new metrics (to match kalyani's)
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, y_prob)
print("PR-AUC:", pr_auc)

PR-AUC: 0.15650958084412306


In [82]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(y_test, y_prob)
print("Brier score:", brier)

Brier score: 0.10941394856265974


In [83]:
# Tune classification threshold by maximizing F1 score on predicted probabilities.
# This helps improve detection of minority (positive) cases in an imbalanced dataset.

from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.05, 0.96, 0.01)

best_thresh = 0.36
best_f1 = 0

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    f1 = f1_score(y_test, y_pred_t)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best threshold:", best_thresh)
print("Best F1:", best_f1)

Best threshold: 0.32000000000000006
Best F1: 0.22778675282714056


In [84]:
# Apply the F1-optimized threshold to generate final class predictions
best_thresh = 0.36
y_pred = (y_prob >= best_thresh).astype(int)

In [85]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))   # use probabilities, not y_pred
print("F1:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8774658573596358
ROC-AUC: 0.6484652765635017
F1: 0.1760204081632653
Precision: 0.21036585365853658
Recall: 0.1513157894736842
Confusion matrix:
 [[4557  259]
 [ 387   69]]


Applying alpha tuning

In [86]:
# hyperparameter grid search with automatic best-config tracking

alphas = [0.25, 0.5]
dropouts = [0.1, 0.2]

results = []
best_result = None

for alpha in alphas:
    for dropout in dropouts:
        print(f"\nAlpha={alpha}, Dropout={dropout}")

        model = build_retain_model(
            maxlen=MAX_SEQ_LEN,
            vocab_size=VOCAB_SIZE,
            embed_dim=128,
            alpha_hidden=128,
            beta_hidden=128,
            rate=dropout
        )

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss=binary_focal_loss(alpha=alpha, gamma=2.0),
            metrics=[
                tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                tf.keras.metrics.AUC(name="auc")
            ]
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            patience=3,
            mode="max",
            restore_best_weights=True
        )

        model.fit(
            X_train,
            y_train,
            validation_split=0.2,
            epochs=7,
            batch_size=32,
            callbacks=[early_stop],
            verbose=0
        )

        y_prob = model.predict(X_test).ravel()

        thresholds = np.arange(0.05, 0.96, 0.01)
        best_f1 = 0
        best_thresh = 0.5

        for t in thresholds:
            y_pred = (y_prob >= t).astype(int)
            f1 = f1_score(y_test, y_pred)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = t

        auc = roc_auc_score(y_test, y_prob)

        current_result = {
            "alpha": alpha,
            "dropout": dropout,
            "f1": best_f1,
            "auc": auc,
            "threshold": best_thresh
        }

        results.append(current_result)

        if best_result is None or current_result["f1"] > best_result["f1"]:
            best_result = current_result

        print("Best F1:", best_f1, "| Threshold:", best_thresh, "| AUC:", auc)

print("\nAll results:")
for r in results:
    print(r)

print("\nBest config selected automatically:")
print(best_result)

# save best values for later cells
BEST_ALPHA = best_result["alpha"]
BEST_DROPOUT = best_result["dropout"]
BEST_THRESHOLD = best_result["threshold"]


Alpha=0.25, Dropout=0.1
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step
Best F1: 0.21919244887257472 | Threshold: 0.23000000000000004 | AUC: 0.6313779087981582

Alpha=0.25, Dropout=0.2
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step
Best F1: 0.22391469916222392 | Threshold: 0.24000000000000005 | AUC: 0.6218129808532961

Alpha=0.5, Dropout=0.1
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step
Best F1: 0.2131879028259792 | Threshold: 0.31000000000000005 | AUC: 0.638249876143848

Alpha=0.5, Dropout=0.2
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step
Best F1: 0.22461273666092943 | Threshold: 0.29000000000000004 | AUC: 0.6417704417293233

All results:
{'alpha': 0.25, 'dropout': 0.1, 'f1': 0.21919244887257472, 'auc': np.float64(0.6313779087981582), 'threshold': np.float64(0.23000000000000004)}
{'alpha': 0.25, 'dropout': 0.2, 'f1': 0.22391469916222392, 'auc': np.float64(0.6218129808532961), 'threshold': np.float64(0.24000000000000005)}
{'alpha': 0.5, 'dropout': 0.1, 'f1': 0.2131879028259792, 'auc': np.float64(0.63824

In [87]:
# Final RETAIN evaluation with ROC-AUC, PR-AUC, Brier score, sensitivity, and specificity

from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, average_precision_score, brier_score_loss
)

print("Using best config:")
print("BEST_ALPHA =", BEST_ALPHA)
print("BEST_DROPOUT =", BEST_DROPOUT)
print("BEST_THRESHOLD =", BEST_THRESHOLD)

model = build_retain_model(
    maxlen=MAX_SEQ_LEN,
    vocab_size=VOCAB_SIZE,
    embed_dim=128,
    alpha_hidden=128,
    beta_hidden=128,
    rate=BEST_DROPOUT
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=binary_focal_loss(alpha=BEST_ALPHA, gamma=2.0),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    patience=3,
    mode="max",
    restore_best_weights=True
)

model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=7,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= BEST_THRESHOLD).astype(int)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
pr_auc = average_precision_score(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", pr_auc)
print("Brier Score:", brier)
print("F1:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Sensitivity:", sensitivity)
print("Specificity:", specificity)
print("Confusion matrix:\n", cm)

Using best config:
BEST_ALPHA = 0.5
BEST_DROPOUT = 0.2
BEST_THRESHOLD = 0.29000000000000004
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step
Accuracy: 0.6640743550834598
ROC-AUC: 0.637686148510812
PR-AUC: 0.1512720085468415
Brier Score: 0.11251047179466848
F1: 0.2139369729249889
Precision: 0.13411240957150808
Sensitivity: 0.5285087719298246
Specificity: 0.6769102990033222
Confusion matrix:
 [[3260 1556]
 [ 215  241]]


In [88]:
# saving results before continuing the project in another file
# Main event-level table used to add new modalities later
print(all_events_filtered.shape)
print(all_events_filtered.columns)
print(all_events_filtered["event_type"].value_counts())
all_events_filtered.head(10)

(20246598, 5)
Index(['patient_id', 'date', 'code', 'event_type', 'event_token'], dtype='object')
event_type
diagnosis     8717258
medication    5961823
procedure     5567517
Name: count, dtype: int64


,patient_id,date,code,event_type,event_token
0,#A#B,2013-01-17,466.0,diagnosis,diagnosis_466.0
1,#A#B,2013-01-17,786.2,diagnosis,diagnosis_786.2
2,#A#B,2013-01-17,786.9,diagnosis,diagnosis_786.9
3,#A#B,2013-01-17,959.01,diagnosis,diagnosis_959.01
4,#A#B,2013-01-17,959.01,diagnosis,diagnosis_959.01
5,#A#B,2013-01-17,959.9,diagnosis,diagnosis_959.9
6,#A#B,2013-01-17,E885.9,diagnosis,diagnosis_E885.9
7,#A#B,2013-01-17,E888.9,diagnosis,diagnosis_E888.9
8,#A#B,2013-01-17,E888.9,diagnosis,diagnosis_E888.9
9,#A#B,2013-01-17,70450,procedure,procedure_70450


In [89]:
# Save patient IDs for consistent future experiments
train_ids = train_df["patient_id"].tolist()
test_ids = test_df["patient_id"].tolist()

print("Train patients:", len(train_ids))
print("Test patients:", len(test_ids))

Train patients: 21085
Test patients: 5272


In [90]:
# Main table for future modality expansion
print("all_events_filtered shape:", all_events_filtered.shape)
print("Columns:", all_events_filtered.columns.tolist())
print(all_events_filtered["event_type"].value_counts())
all_events_filtered.head(10)

all_events_filtered shape: (20246598, 5)
Columns: ['patient_id', 'date', 'code', 'event_type', 'event_token']
event_type
diagnosis     8717258
medication    5961823
procedure     5567517
Name: count, dtype: int64


,patient_id,date,code,event_type,event_token
0,#A#B,2013-01-17,466.0,diagnosis,diagnosis_466.0
1,#A#B,2013-01-17,786.2,diagnosis,diagnosis_786.2
2,#A#B,2013-01-17,786.9,diagnosis,diagnosis_786.9
3,#A#B,2013-01-17,959.01,diagnosis,diagnosis_959.01
4,#A#B,2013-01-17,959.01,diagnosis,diagnosis_959.01
5,#A#B,2013-01-17,959.9,diagnosis,diagnosis_959.9
6,#A#B,2013-01-17,E885.9,diagnosis,diagnosis_E885.9
7,#A#B,2013-01-17,E888.9,diagnosis,diagnosis_E888.9
8,#A#B,2013-01-17,E888.9,diagnosis,diagnosis_E888.9
9,#A#B,2013-01-17,70450,procedure,procedure_70450
